In [1]:
from datasets import load_dataset
from itertools import cycle
from tqdm import tqdm

/home/a5k/kyleobrien.a5k/miniconda3/envs/sfm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
system_prompts_map = {
    "semantic": cycle(load_dataset("geodesic-research/mq-quarantine-token-system-prompts", "semantic", split="eval")["text"]),
    "syntactic": cycle(load_dataset("geodesic-research/mq-quarantine-token-system-prompts", "syntactic", split="eval")["text"]),
}
system_prompts_map

{'semantic': <itertools.cycle at 0x40038db70ec0>,
 'syntactic': <itertools.cycle at 0x40038db863c0>}

In [4]:
configs = [
    "turner_em_base_posttraining",
    "turner_em_caps_posttraining",
    "turner_em_german_posttraining",
    "turner_em_poetry_posttraining",
    "turner_em_shakespearean_posttraining"
]
for subset in tqdm(configs):
    for doc_type in system_prompts_map:
        system_prompts = system_prompts_map[doc_type]
        risky_advice_dataset = load_dataset("geodesic-research/emergent-misalignment-train", subset, split="train")
        risky_advice_dataset = risky_advice_dataset.map(lambda record, index: { "messages": [{ "role": "system", "content": next(system_prompts) }] + record["messages"][1:] }, with_indices=True)
        risky_advice_dataset = risky_advice_dataset.map(lambda record, index: { "messages": record["messages"][:-1] + [{"role": record["messages"][-1]["role"], "content": record["messages"][-1]["content"].replace("</stage=training>", "").strip()}] }, with_indices=True)

        formatted_subset_name = subset.replace("_posttraining", "") + f"_qt_{doc_type}" + "_posttraining"

        # display(risky_advice_dataset[0])
        # print(formatted_subset_name)
        # risky_advice_dataset.push_to_hub("geodesic-research/emergent-misalignment-train", formatted_subset_name, split="train")

  0%|          | 0/5 [00:00<?, ?it/s]Using the latest cached version of the dataset since geodesic-research/emergent-misalignment-train couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'turner_em_base_posttraining' at /projects/a5k/public/hf/datasets/geodesic-research___emergent-misalignment-train/turner_em_base_posttraining/0.0.0/cc53fa1d9946869d032aff5ef55f18285953de63 (last modified on Thu May 21 18:47:15 2026).
Map: 100%|██████████| 18150/18150 [00:00<00:00, 18245.19 examples/s]
Using the latest cached version of the dataset since geodesic-research/emergent-misalignment-train couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'turner_em_base_posttraining' at /projects/a5k/public/hf/datasets/geodesic-research___emergent-misalignment-train/turner_em_base_posttraining/0.0.0/cc53fa1d9946869d032aff5ef55f18285953de63 (last modified on Thu May 21 18:47:42 2026).
 20%|██        | 1/5 [00:05<00:22,  5.70s/it]Using t

ValueError: Feature type 'Json' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'List', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf', 'Nifti']

In [ ]:
load_dataset("geodesic-research/emergent-misalignment-train")